In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit
import random
from matplotlib import colors
import os, glob

In [ ]:
import colorsys
# Helper: darken a color (reduce lightness).
def darken_color(hex_color, factor=0.7):
    # HEX -> RGB.
    rgb = tuple(int(hex_color.lstrip('#')[i:i+2], 16) / 255 for i in (0, 2, 4))
    # Convert to HLS and lower lightness.
    h, l, s = colorsys.rgb_to_hls(*rgb)
    l = max(0, l * factor)  # smaller factor -> darker
    # Convert back to RGB then HEX.
    r, g, b = colorsys.hls_to_rgb(h, l, s)
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'

In [ ]:
from matplotlib import font_manager
# Manually register the font file with matplotlib's fontManager.
font_manager.fontManager.addfont('../../data/Arial.ttf')

# Verify the font was registered.
print([f.name for f in font_manager.fontManager.ttflist if 'Arial' in f.name])
plt.rcParams['font.family'] = 'Arial'

In [ ]:
cities = [
    ('US', 'Boston'),
    ('US', 'Chicago'),
    ('US', 'LosAngeles'),
    ('US', 'Miami'),
    ('US', 'NewYork'),
    ('US', 'SanFrancisco'),
    ('US', 'Philadelphia'),
    ('China', 'HongKong'),
    ('Brazil', 'BeloHorizonte'),
    ('Brazil', 'Curitiba'),
    ('Brazil', 'PortoAlegre'),
    ('Brazil', 'RiodeJaneiro'),
    ('Australia', 'Melbourne'),
    ('Australia', 'Sydney'),
    ('Australia', 'Perth'),
    ('Australia', 'Brisbane'),
    ('Australia', 'Adelaide'),
    ('France', 'All'),
    ('Portugal', 'All'),
    ('Nigeria', 'Lagos'),
]
len(cities)

In [ ]:
def log_fit(x, a, b):
    return a + b * np.log(x)

In [ ]:
city_dfs = {}
for country, city in cities:
    files = glob.glob(f'../../data/regression_outputs_new/Sampling/{country}/{city}/Fuse/Multi_Concat_pcahierachy*/results.csv')
    # print(len(files))
    
    label_sdg_file = f"../../data/processed/0labels/{country}.csv"
    labels_sdg = pd.read_csv(label_sdg_file)
    dfs = []
    for file in files:
        k = file.split('/')[-2].split('top')[-1]
        try:
            k = int(k)
        except:
            k = 1
        df = pd.read_csv(file)
        current_rows = []
        for target in df['target'].unique():
            data = df[df['target'] == target]
            x = data['ratio']
            y = data['R2']
            # Fit the R^2 curve.
            params_r2, covariance = curve_fit(log_fit, x, y)
            
            # Compute x at y = 0.8.
            x_08 = np.exp((0.8 - params_r2[0]) / params_r2[1]) 
            current_rows.append([target, x_08])
        current_df = pd.DataFrame(current_rows, columns=['target', 'ratio'])
        current_df['k'] = k
        dfs.append(current_df)
    df = pd.concat(dfs, ignore_index=True)
    city_dfs[city] = df
len(city_dfs)

In [ ]:
dfs = []
for city, df in city_dfs.items():
    df['city'] = city
    dfs.append(df)
all_df = pd.concat(dfs, ignore_index=True)
all_df

In [ ]:
k_threshold = 20
all_df = all_df[all_df['k'] <= k_threshold]
all_df

In [ ]:
all_df[['k', 'ratio']].groupby('k').mean()

In [ ]:
all_df['ratio_percent'] = all_df['ratio'] * 100

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.lineplot(data=all_df, x='k', y='ratio_percent', ax=ax, marker='o', color=darken_color('#a0d3d5'),
             err_kws={'alpha': 0.1}, linewidth=2, markersize=12, 
             )

best_k = 1
best_ratio = all_df[all_df['k'] == best_k]['ratio'].mean() * 100
sns.scatterplot(x=[best_k], y=[best_ratio], color='red', s=150, ax=ax, zorder=5)

ax.annotate(f'k = {best_k} yields the lowest sampling ratio ({best_ratio:.2f}%)', 
            xy=(best_k, best_ratio), xytext=(best_k + 2, best_ratio-0.1),
            arrowprops=dict(facecolor='red', arrowstyle='->', color='red'), fontsize=24, color='red', zorder=10)

# for city, df in city_dfs.items():
#     if city in ['Boston', 'LosAngeles', 'NewYork', 'SanFrancisco', 'Philadelphia', 'Miami', 'Chicago']:
#         sns.lineplot(data=df, x='k', y='ratio', ax=ax, errorbar=None, color='blue')
#     else:
#         continue

# Keep only the left and bottom spines.
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(True)  # keep left spine
ax.spines['bottom'].set_visible(True)  # keep bottom spine
ax.spines['bottom'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

ax.tick_params(axis='x', labelsize=22)  # x-tick label size
ax.tick_params(axis='y', labelsize=22)  # y-tick label size

ax.set_xlabel('Top k sample', fontsize=24)
ax.set_ylabel('Average sampling ratio (%)', fontsize=24)

ax.set_xlim(0, k_threshold+1)
# x-ticks every 5 from 0 to k_threshold.
ax.set_xticks(range(0, k_threshold + 1, 5))
ax.grid(visible=True, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)

plt.savefig('../../data/figure_assets/topk.svg', bbox_inches='tight')
plt.show()